In [2]:
import numpy as np
from pathlib import Path


%load_ext autoreload
%autoreload 2

In [ ]:
# 1.000000 1169.665771 142.498505 1488.142334 992.517578 0.945230 0.000000
# 1.000000 711.329224 199.139221 886.791992 722.951721 0.932172 0.000000
# 1.000000 935.158875 218.925522 1077.071411 665.722900 0.916619 0.000000
# 1.000000 600.925232 306.522552 704.481018 571.216736 0.908921 0.000000
# 1.000000 554.193726 307.945465 618.682434 511.767303 0.885185 0.000000
# 1.000000 1047.593628 299.711700 1134.147461 554.448486 0.882793 0.000000
# 1.000000 865.318604 299.371704 953.406616 540.172791 0.869008 0.000000
# 1.000000 1123.249023 314.787476 1201.708618 562.013672 0.862366 0.000000
# 1.000000 691.353699 262.257599 796.445190 565.606995 0.861965 0.000000

In [98]:
dir_path = Path("/home/kozakmv/Programming/Python/boxmot/runs/dets_n_embs/yolox_m_strongsort")

In [99]:
def from_npy_to_dets_embs(npy_path: Path):
    npy_arr = np.load(npy_path)

    new_name = npy_path.stem[:npy_path.stem.rfind("-")]

    dets_path = npy_path.parent / "dets" / (new_name + ".txt")
    embs_path = npy_path.parent / "embs"/ (new_name + ".txt")

    dets_path.parent.mkdir(parents=True, exist_ok=True)
    embs_path.parent.mkdir(parents=True, exist_ok=True)

    new_dets = []
    new_embs = []
    for i in range(npy_arr.shape[0]):
        cur_row = npy_arr[i]
        dets, embs = cur_row[:10], cur_row[10:]
        new_det = [dets[0], dets[2], dets[3], dets[2] + dets[4], dets[3] + dets[5], dets[6], 0]
        
        new_dets.append(new_det)
        new_embs.append(embs)
    
    np.savetxt(dets_path, np.array(new_dets), header=f"tracking/val_utils/data/MOT17_half/train/{new_name}/img1", fmt ='%f6')
    np.savetxt(embs_path, np.array(new_embs), fmt ='%f6')

In [5]:
def xyxy2xywh(x):
    """
    Convert bounding box coordinates from (x1, y1, x2, y2) format to (x, y, width, height) format.

    Args:
        x (np.ndarray) or (torch.Tensor): The input bounding box coordinates in (x1, y1, x2, y2) format.
    Returns:
       y (np.ndarray) or (torch.Tensor): The bounding box coordinates in (x, y, width, height) format.
    """
    y = np.copy(x)
    y[..., 0] = (x[..., 0] + x[..., 2]) / 2  # x center
    y[..., 1] = (x[..., 1] + x[..., 3]) / 2  # y center
    y[..., 2] = x[..., 2] - x[..., 0]  # width
    y[..., 3] = x[..., 3] - x[..., 1]  # height
    return y


In [100]:
for file_name in dir_path.glob("*.npy"):
    from_npy_to_dets_embs(file_name)

In [95]:
dir_path = Path("/home/kozakmv/Programming/Python/boxmot/tracking/val_utils/data/MOT17_half/train/")

In [96]:
dets_path = Path("/home/kozakmv/Programming/Python/boxmot/runs/dets_n_embs/yolox_m_strongsort") / "dets"

In [97]:
for dir_name in sorted(list(dir_path.iterdir())):
    dets_file_path =  dets_path / (dir_name.name + ".txt")
    dets_data = np.loadtxt(dets_file_path, skiprows=1)
    min_id = int(dets_data[:, 0].min())

    dets_data[:, 0] = dets_data[:, 0] - min_id + 1

    np.savetxt(dets_file_path, dets_data, header=f"tracking/val_utils/data/MOT17_half/train/{dir_name.name}/img1", fmt="%6f")

    gt_path = dir_name / "gt" / "gt.txt"

    data = np.loadtxt(gt_path, delimiter=",")
    data = data[data[:, 0] >= min_id]

    data[:, 0] = data[:, 0] - min_id + 1

    np.savetxt(gt_path, data, fmt="%6f", delimiter=",")

    old_new_paths: list[tuple[Path, Path]] = []
    img_dir_path = dir_name / "img1"
    for img_path in sorted(img_dir_path.iterdir()):
        if int(img_path.stem) < min_id:
            img_path.unlink()
        else:
            new_name = str(int(img_path.stem) - min_id + 1).zfill(6)
            old_new_paths.append((img_path, img_dir_path / (new_name + img_path.suffix)))

    for old_path, new_path in old_new_paths:
        old_path.rename(new_path)

In [11]:
dets = np.array([[1, 2, 3, 4, 5, 6, 7], [1, 2, 3, 4, 5, 6, 7]])

In [11]:
x = np.array([1, 2, 3, 4, 5, 6, 7])
y = np.array([0, 1, 1, 1, 0, 0, 1])

active = [0, 3, 5, 6]

x[y == 1]

array([2, 3, 4, 7])

In [17]:
x[np.intersect1d(np.where(y == 1), active)]

array([4, 7])

In [ ]:
x[np.intersect1d(np.where(y == 1), active)]

array([4, 7])

In [18]:
from boxmot.utils.iou import AssociationFunction

In [19]:
a = np.array([[10, 30, 40, 70], [20, 50, 60, 90]])
b = np.array([[15, 45, 55, 85], [25, 55, 65, 95]])


iou = AssociationFunction.iou_batch(a, b)
print(iou)

[[0.28735632 0.08737864]
 [0.62025316 0.62025316]]


In [20]:
matches = [(1, 3), (2, 1), (3, 6)]

In [25]:
itracks, idets = list(zip(*matches))

itracks, idets

((1, 2, 3), (3, 1, 6))

In [26]:
dets_high_pool = range(6)

In [37]:
a = np.array([1, 2, 3, 4, 5, 6, 7])
idx = np.zeros(7, dtype=bool)
idx[[1, 3, 6]] = True

states = np.array([0, 0, 3, 3, 3, 3, 3])

a[states==3]

array([3, 4, 5, 6, 7])

In [41]:
idx

array([False,  True, False,  True, False, False,  True])

In [45]:
a[idx]

array([2, 4, 7])

In [43]:
a[idx & (states == 3)]

array([4, 7])

In [ ]:
import numpy as np
import scipy.linalg
from typing import Tuple

"""
Table for the 0.95 quantile of the chi-square distribution with N degrees of
freedom (contains values for N=1, ..., 9). Taken from MATLAB/Octave's chi2inv
function and used as Mahalanobis gating threshold.
"""
chi2inv95 = {
    1: 3.8415,
    2: 5.9915,
    3: 7.8147,
    4: 9.4877,
    5: 11.070,
    6: 12.592,
    7: 14.067,
    8: 15.507,
    9: 16.919
}


class BaseKalmanFilter:
    """
    Base class for Kalman filters tracking bounding boxes in image space.
    """

    def __init__(self, ndim: int):
        self.ndim = ndim
        self.dt = 1.

        # Create Kalman filter model matrices.
        self._motion_mat = np.eye(2 * ndim, 2 * ndim)  # State transition matrix
        for i in range(ndim):
            self._motion_mat[i, ndim + i] = self.dt
        self._update_mat = np.eye(ndim, 2 * ndim)  # Observation matrix

        # Motion and observation uncertainty weights.
        self._std_weight_position = 1. / 20
        self._std_weight_velocity = 1. / 160

    def initiate(self, measurement: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Create track from unassociated measurement.
        """
        mean_pos = measurement
        mean_vel = np.zeros_like(mean_pos)
        mean = np.r_[mean_pos, mean_vel]

        std = self._get_initial_covariance_std(measurement)
        covariance = np.diag(np.square(std))
        return mean, covariance

    def _get_initial_covariance_std(self, measurement: np.ndarray) -> np.ndarray:
        """
        Return initial standard deviations for the covariance matrix.
        Should be implemented by subclasses.
        """
        raise NotImplementedError

    def predict(self, mean: np.ndarray, covariance: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Run Kalman filter prediction step.
        """
        std_pos, std_vel = self._get_process_noise_std(mean)
        motion_cov = np.diag(np.square(np.r_[std_pos, std_vel]))

        mean = np.dot(mean, self._motion_mat.T)
        covariance = np.linalg.multi_dot((
            self._motion_mat, covariance, self._motion_mat.T)) + motion_cov

        return mean, covariance

    def _get_process_noise_std(self, mean: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Return standard deviations for process noise.
        Should be implemented by subclasses.
        """
        raise NotImplementedError

    def project(self, mean: np.ndarray, covariance: np.ndarray, confidence: float = 0.0) -> Tuple[np.ndarray, np.ndarray]:
        """
        Project state distribution to measurement space.
        """
        std = self._get_measurement_noise_std(mean, confidence)
        
        # NSA Kalman algorithm from GIAOTracker, which proposes a formula to 
        # adaptively calculate the noise covariance Rek:
        # Rk = (1 − ck) Rk
        # where Rk is the preset constant measurement noise covariance
        # and ck is the detection confidence score at state k. Intuitively,
        # the detection has a higher score ck when it has less noise,
        # which results in a low Re.
        std = [(1 - confidence) * x for x in std]
        
        innovation_cov = np.diag(np.square(std))

        mean = np.dot(self._update_mat, mean)
        covariance = np.linalg.multi_dot((
            self._update_mat, covariance, self._update_mat.T))
        return mean, covariance + innovation_cov

    def multi_predict(self, mean: np.ndarray, covariance: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Run Kalman filter prediction step (Vectorized version).
        """
        std_pos, std_vel = self._get_multi_process_noise_std(mean)
        sqr = np.square(np.r_[std_pos, std_vel]).T

        motion_cov = [np.diag(sqr[i]) for i in range(len(mean))]
        motion_cov = np.asarray(motion_cov)

        mean = np.dot(mean, self._motion_mat.T)
        left = np.dot(self._motion_mat, covariance).transpose((1, 0, 2))
        covariance = np.dot(left, self._motion_mat.T) + motion_cov

        return mean, covariance

    def update(self, mean: np.ndarray, covariance: np.ndarray, measurement: np.ndarray, confidence: float = 0.0) -> Tuple[np.ndarray, np.ndarray]:
        """
        Run Kalman filter correction step.
        """
        projected_mean, projected_cov = self.project(mean, covariance, confidence)

        chol_factor, lower = scipy.linalg.cho_factor(projected_cov, lower=True, check_finite=False)
        kalman_gain = scipy.linalg.cho_solve((chol_factor, lower), np.dot(covariance, self._update_mat.T).T, check_finite=False).T
        innovation = measurement - projected_mean

        new_mean = mean + np.dot(innovation, kalman_gain.T)
        new_covariance = covariance - np.linalg.multi_dot((kalman_gain, projected_cov, kalman_gain.T))
        return new_mean, new_covariance
    
    def multi_update(self, means: np.ndarray, covs: np.ndarray, measurements: np.ndarray, confs: np.ndarray = None) -> Tuple[np.ndarray, np.ndarray]:
        """
        Run Kalman filter correction step (Vectorized version).
        """
        

    def _get_multi_process_noise_std(self, mean: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Return standard deviations for process noise in vectorized form.
        Should be implemented by subclasses.
        """
        raise NotImplementedError

    def gating_distance(self, mean: np.ndarray, covariance: np.ndarray, measurements: np.ndarray, only_position: bool = False, metric: str = 'maha') -> np.ndarray:
        """
        Compute gating distance between state distribution and measurements.
        """
        mean, covariance = self.project(mean, covariance)

        if only_position:
            mean, covariance = mean[:2], covariance[:2, :2]
            measurements = measurements[:, :2]

        d = measurements - mean
        if metric == 'gaussian':
            return np.sum(d * d, axis=1)
        elif metric == 'maha':
            cholesky_factor = np.linalg.cholesky(covariance)
            z = scipy.linalg.solve_triangular(cholesky_factor, d.T, lower=True, check_finite=False, overwrite_b=True)
            squared_maha = np.sum(z * z, axis=0)
            return squared_maha
        else:
            raise ValueError('invalid distance metric')

In [7]:
tracks = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
track_pool = np.zeros(10, dtype=bool)
track_pool[1:5] = 1
matches = [0, 2, 3]

In [22]:
def subset_pool(track_pool: np.ndarray, indices: np.ndarray) -> np.ndarray:
    """
    Create a subset of the track pool.

    Example:
    track_pool == [0, 0, 1, 1, 1, 0]
    indices = [0, 2]
    result == [0, 0, 1, 0, 1, 0]
    """
    new_pool = np.zeros_like(track_pool)
    new_pool[np.where(track_pool)[0][indices]] = 1
    return new_pool


array([False, False,  True, False,  True, False])

In [19]:
new_pool = np.zeros_like(track_pool)
new_pool[np.where(track_pool)[0][matches]] = 1

tracks[new_pool]

array([2, 4, 5])

In [20]:
tracks[track_pool][matches]

array([2, 4, 5])

In [4]:
def mask_from_ids(arr: np.ndarray, ids: np.ndarray):
    mask = np.zeros(arr.shape[0], dtype=bool)
    mask[ids] = True
    return mask

def mask_from_values(arr: np.ndarray, values: np.ndarray):
    mask = np.zeros(arr.shape[0], dtype=bool)
    mask[np.isin(arr, values)] = True
    return mask

def subset_pool(pool: np.ndarray, indices: np.ndarray) -> np.ndarray:
    """
    Create a subset of the track pool.

    Example:
    pool == [0, 0, 1, 1, 1, 0]
    indices = [0, 2]
    result == [0, 0, 1, 0, 1, 0]
    """
    new_pool = np.zeros_like(pool)
    new_pool[np.where(pool)[0][indices]] = 1
    return new_pool

In [8]:
t_ids = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
d_ids = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])

track_pool = np.array([1, 1, 1, 1, 1, 1, 1, 0, 0, 0], dtype=bool)
tracked_pool = np.array([0, 0, 0, 1, 0, 0, 1, 0, 0, 0], dtype=bool)

det_pool = np.array([0, 0, 0, 1, 1, 1, 1, 1, 1], dtype=bool)

track_m = np.array([1, 2, 4, 6])
det_m = np.array([3, 1, 5, 2])

track_m_pool = subset_pool(track_pool, track_m)
det_m_pool = subset_pool(det_pool, det_m)

upd_pool = track_m_pool & tracked_pool
react_pool = track_m_pool & ~tracked_pool

In [7]:
# Create a mapping from track_m to det_m
mapping = np.zeros_like(track_pool, dtype=int)
mapping[track_m] = det_m

# Filter det_m_pool based on upd_pool
upd_det_pool = np.zeros_like(det_pool, dtype=bool)
upd_det_pool[mapping[upd_pool]] = True

# Filter det_m_pool based on react_pool
react_det_pool = np.zeros_like(det_pool, dtype=bool)
react_det_pool[mapping[react_pool]] = True

print("upd_det_pool:", upd_det_pool)
print("react_det_pool:", react_det_pool)

upd_det_pool: [False False  True False False False False False False]
react_det_pool: [False  True False  True False  True False False False]


In [41]:
upd_pool

array([False, False, False, False, False, False,  True, False, False,
       False])

In [58]:
a = np.array([1, 2, 3])